### 1.

In [6]:
import pandas as pd

loans = pd.read_csv('data/library_loans_clean.csv', parse_dates=['loan_date', 'return_date'])


print(loans.shape)
print(loans.dtypes)

members_raw = pd.read_csv('data/library_members.csv') 
members = members_raw.copy()


print(loans.isna().sum())
print((loans.isna().mean() * 100).round(1))

(600, 8)
loan_id                   str
member_id                 str
title                     str
genre                     str
loan_date      datetime64[us]
return_date    datetime64[us]
late_fee              float64
returned                 bool
dtype: object
loan_id         0
member_id       0
title           0
genre           0
loan_date       3
return_date    36
late_fee        0
returned        0
dtype: int64
loan_id        0.0
member_id      0.0
title          0.0
genre          0.0
loan_date      0.5
return_date    6.0
late_fee       0.0
returned       0.0
dtype: float64


### 2.

In [ ]:
print(loans['loan_date'].isna().sum())


print(loans['return_date'].isna().sum())

print(loans['returned'].isna().sum())  

print(loans['returned'].isna().sum())  



3
36
     loan_id member_id                      title     genre  loan_date  \
52   L-30053     M0064      The Brave Little Boat  Children 2026-01-21   
68   L-30069     M0068        Shadow of the Tower   Fantasy 2026-01-25   
90   L-30091     M0029            A Quiet Witness     Crime 2026-02-01   
91   L-30092     M0144     Night Train to Nowhere     Crime 2026-02-01   
93   L-30094     M0126     Wolves of the Old Wood   Fantasy 2026-01-31   
100  L-30101     M0026         The Silent Harbour     Crime 2026-02-03   
104  L-30105     M0110          Between the Lines   Romance 2026-02-05   
110  L-30111     M0101          Between the Lines   Romance 2026-02-05   
128  L-30129     M0027              Salt and Iron   History 2026-02-11   
139  L-30140     M0119       Ollie Learns to Swim  Children 2026-02-15   
141  L-30142     M0029         Mimmi and the Moon  Children 2026-02-15   
151  L-30152     M0105          Between the Lines   Romance 2026-02-18   
175  L-30176     M0032  The Drago

In [14]:
loans['has_return_date'] = loans['return_date'].notna()

loans['has_return_date'].value_counts()

has_return_date
True     564
False     36
Name: count, dtype: int64

### 3.


In [18]:
members_raw = pd.read_csv('data/library_members.csv') 
members = members_raw.copy()

print(members.shape)

members['birth_year'].info()
members['birth_year'].describe()

(148, 3)
<class 'pandas.Series'>
RangeIndex: 148 entries, 0 to 147
Series name: birth_year
Non-Null Count  Dtype  
--------------  -----  
130 non-null    float64
dtypes: float64(1)
memory usage: 1.3 KB


count     130.000000
mean     1983.384615
std        25.158223
min      1900.000000
25%      1978.250000
50%      1988.000000
75%      1996.750000
max      2030.000000
Name: birth_year, dtype: float64

In [23]:
print((members['birth_year'] == 1900).sum())
print((members['birth_year'] == 2030).sum())

8
3


In [ ]:
print(members.duplicated().sum())

print(members['member_id'].duplicated().sum())

# 8 members have too small birth_year
# 3 members have too large birth_year
# 4 rows are duplicated

4
4


### 4.

In [27]:
members['birth_year'] = members['birth_year'].where(members['birth_year'].between(1920, 2015))

members['birth_year'].describe()


count     119.000000
mean     1987.815126
std        12.062666
min      1957.000000
25%      1979.000000
50%      1988.000000
75%      1996.500000
max      2010.000000
Name: birth_year, dtype: float64

### 5.

In [28]:
print(members.dropna().shape)# Any column
print(members.dropna(subset=['branch']).shape) # Branch is missing
print(members.dropna(subset=['birth_year']).shape) # Birth year is missing  


(107, 3)
(136, 3)
(119, 3)


In [29]:
members['branch'] = members['branch'].fillna('Unknown')

members['branch'].value_counts()

branch
Central    40
South      34
North      32
East       17
Harbour    13
Unknown    12
Name: count, dtype: int64

In [31]:
birth_year_median = members['birth_year'].median()
birth_year_filled = members['birth_year'].fillna(birth_year_median)

print(birth_year_median)
print(members['birth_year'].mean().round(2), birth_year_filled.mean().round(2))
print(members['birth_year'].std().round(2), birth_year_filled.std().round(2))

1988.0
1987.82 1987.85
12.06 10.81


### 6.

In [33]:
merged = pd.merge(loans, members, on='member_id', how='left')

merged.shape

(616, 11)

In [34]:
merged[merged['member_id'].duplicated(keep=False)].sort_values('member_id')

,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned,has_return_date,branch,birth_year
380,L-30373,M0001,Between the Lines,Romance,2026-04-23,2026-05-11,0.0,True,True,Central,1971.0
385,L-30378,M0001,Cold Case 1987,Crime,2026-04-25,2026-05-24,20.0,True,True,Central,1971.0
517,L-30508,M0001,Cold Case 1987,Crime,2026-06-02,2026-06-08,0.0,True,True,Central,1971.0
262,L-30258,M0001,The Brave Little Boat,Children,2026-03-20,2026-04-26,40.0,True,True,Central,1971.0
454,L-30446,M0001,Between the Lines,Romance,2026-05-15,2026-06-22,42.5,True,True,Central,1971.0
...,...,...,...,...,...,...,...,...,...,...,...
47,L-30047,M0150,The Ember Crown,Fantasy,2026-01-19,2026-02-28,47.5,True,True,Central,1994.0
424,L-30416,M0150,Between the Lines,Romance,2026-05-05,2026-06-11,40.0,True,True,Central,1994.0
552,L-30541,M0150,Cold Case 1987,Crime,2026-06-10,2026-06-24,0.0,True,True,Central,1994.0
259,L-30255,M0150,A Quiet Witness,Crime,2026-03-20,2026-04-04,0.0,True,True,Central,1994.0


In [36]:
left = loans.merge(members, on='member_id', how='left', validate='many_to_one')

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 member_id
    M0015
    M0033
    M0065
    M0071 ...

In [43]:
members = members.drop_duplicates().reset_index(drop=True)

left = loans.merge(members, on='member_id', how='left', validate='many_to_one')

print(left.shape)
print(left['late_fee'].sum())


(600, 11)
7487.5


### 8.

In [46]:
merged['branch'] = merged['branch'].fillna('No member record')

merged.groupby('branch')['late_fee'].count().sort_values(ascending=False)

branch
Central              154
South                136
North                126
East                  72
Harbour               51
Unknown               46
No members record     31
Name: late_fee, dtype: int64